# FreightDesk: A LangChain Agent, Built Step by Step

**What you build:** an air cargo exception desk assistant that looks up a shipment, checks a flight,
finds alternates, and asks a human before it moves anything.

**How you build it:** naive first, then fix what breaks. Each fix creates the next problem.
That chain is the lesson.

| | |
|---|---|
| **Steps** | 16, each one runnable on its own |
| **Every step has** | a flowchart, an idea in plain words, code that runs, what to notice, one production fact |
| **Needs** | Amazon Bedrock model access, or nothing at all if you set `MODE = "offline"` |
| **Diagrams** | plain ASCII, so they render in Colab, VS Code, GitHub and nbviewer alike |

### The case you are working

> AWB **160-45872910**. Pharma, 2 to 8 C, 480 kg, 12 pieces.
> Booked **MA-217 BLR to AMS on 6 Aug**. Flight cancelled.
> Shipper has about 30 hours of temperature reserve.
> Contact `ops@kavery-exports.example` / `+91 98450 11223`.

### How to run

**VS Code**
1. `python -m venv .venv` then activate it (Windows: `.venv\Scripts\activate`)
2. `pip install -U langchain langchain-aws langgraph`
3. `aws configure`, or export `AWS_ACCESS_KEY_ID`, `AWS_SECRET_ACCESS_KEY`, `AWS_DEFAULT_REGION`
4. Select the venv as the notebook kernel, then Run All

**Google Colab**
1. Run the install cell below
2. Set the three AWS variables in the config cell, or leave `MODE = "offline"`
3. Runtime, Run all

### Three rules for reading this notebook

1. **Run every cell in order.** Later steps use variables from earlier ones.
2. **Read the output, not just the code.** Half the lessons are in what the trace shows.
3. **When unsure of a parameter name, introspect it.** `inspect.signature(X.__init__)`. Never guess.

In [ ]:
# Colab only. On a local venv, install from a terminal instead.
# !pip install -q -U langchain langchain-aws langgraph

---
# 1. Configuration

Every knob in this notebook lives in one cell. Nothing below hardcodes a value.

That is not tidiness. It is the first production habit: a config surface you can diff, review and
change per environment without touching logic.

| Group | What it controls |
|---|---|
| 1 Mode | Live Bedrock or a deterministic offline stand-in |
| 2 Model | Model id, region, sampling |
| 3 Retry | How transient failures are handled |
| 4 Approval | Which tools need a human decision |
| 5 Privacy | What gets redacted and where |
| 6 Compaction | When a long thread gets summarized |
| 7 Limits | Cost and loop ceilings |
| 8 Observability | Trace verbosity |

In [ ]:
# ================================================================
# CONFIGURATION
# ================================================================

# --- 1. MODE ---------------------------------------------------
MODE = "offline"          # "bedrock" = live Amazon Bedrock
                          # "offline" = deterministic scripted model, no AWS, no cost

# --- 2. MODEL --------------------------------------------------
MODEL_ID    = "us.anthropic.claude-haiku-4-5-20251001-v1:0"   # "us." prefix is required
REGION      = "us-east-1"
TEMPERATURE = 0           # set temperature OR top_p on Claude 4.x, never both
MAX_TOKENS  = 1024

# --- 3. RETRY --------------------------------------------------
RETRY_MAX_ATTEMPTS  = 2        # retries after the first try, so 3 attempts total
RETRY_INITIAL_DELAY = 0.5      # seconds
RETRY_BACKOFF       = 2.0
RETRY_TOOLS         = ["get_flight_status"]     # scope it, never leave it global
CARRIER_FLAKY_UNTIL = 3        # simulated: the status API fails on calls 1 and 2

# --- 4. APPROVAL GATE ------------------------------------------
GATE_ENABLED   = True
GATE_ON        = {
    "rebook_shipment":  ["approve", "edit", "reject"],   # small, enumerable args, editing is safe
    "notify_customer":  ["approve", "reject"],           # free text, no in-popup editing
}
GATE_PREFIX    = "FreightDesk action pending approval"

# --- 5. PRIVACY ------------------------------------------------
PII_EMAIL_STRATEGY = "redact"      # block | redact | mask | hash
PII_PHONE_STRATEGY = "mask"
PII_PHONE_REGEX    = r"\+\d{1,3}[\s.-]?\d{4,5}[\s.-]?\d{4,6}"   # leading + required, see Step 10
PII_ON_INPUT        = True
PII_ON_TOOL_RESULTS = True         # the door most people forget
PII_ON_OUTPUT       = False

# --- 6. COMPACTION ---------------------------------------------
COMPACT_ENABLED          = True
COMPACT_TRIGGER_MESSAGES = 12      # summarize once history reaches this many messages
COMPACT_KEEP_MESSAGES    = 6       # keep this many recent messages verbatim

# --- 7. LIMITS -------------------------------------------------
LIMIT_TOOL_NAME  = "find_alternate_flights"
LIMIT_RUN_CALLS  = 3               # per user turn
LIMIT_MODEL_RUN  = 8               # model calls per user turn, stops runaway loops

# --- 8. OBSERVABILITY ------------------------------------------
TRACE = True                       # print a structured line per model call

# --- Case under work -------------------------------------------
CASE_AWB  = "160-45872910"
CASE_TEXT = ("MA-217 on 6 Aug is cancelled. AWB 160-45872910. Pharma cold chain, "
             "about 30 hours of reserve. Contact ops@kavery-exports.example "
             "or +91 98450 11223. What do we do?")

In [ ]:
# Print the effective configuration. Do this in production too: a service that cannot
# tell you what settings it is running under cannot be debugged at 2am.
_groups = {
    "1 Mode":          ["MODE"],
    "2 Model":         ["MODEL_ID", "REGION", "TEMPERATURE", "MAX_TOKENS"],
    "3 Retry":         ["RETRY_MAX_ATTEMPTS", "RETRY_INITIAL_DELAY", "RETRY_BACKOFF", "RETRY_TOOLS"],
    "4 Approval":      ["GATE_ENABLED", "GATE_ON", "GATE_PREFIX"],
    "5 Privacy":       ["PII_EMAIL_STRATEGY", "PII_PHONE_STRATEGY", "PII_ON_INPUT",
                        "PII_ON_TOOL_RESULTS", "PII_ON_OUTPUT"],
    "6 Compaction":    ["COMPACT_ENABLED", "COMPACT_TRIGGER_MESSAGES", "COMPACT_KEEP_MESSAGES"],
    "7 Limits":        ["LIMIT_TOOL_NAME", "LIMIT_RUN_CALLS", "LIMIT_MODEL_RUN"],
    "8 Observability": ["TRACE"],
}
for group, keys in _groups.items():
    print(f"\n[{group}]")
    for k in keys:
        print(f"   {k:26s} = {globals()[k]}")

---
# 2. Preflight

Never assume a library version. Ask it.

Two things go wrong in agent projects more than any other: a parameter that was renamed, and a
class that does not exist yet on your build. Both are cheap to detect and expensive to discover
halfway through a demo.

In [ ]:
from importlib.metadata import version

for pkg in ["langchain", "langchain-core", "langgraph", "langchain-aws"]:
    try:
        print(f"{pkg:16s} {version(pkg)}")
    except Exception:
        print(f"{pkg:16s} not installed")

import langchain.agents.middleware as _mw
print()
for name in ["HumanInTheLoopMiddleware", "PIIMiddleware", "SummarizationMiddleware",
             "ToolRetryMiddleware", "ToolCallLimitMiddleware", "ModelCallLimitMiddleware",
             "ToolErrorMiddleware", "wrap_tool_call", "before_model", "after_model"]:
    print(f"{name:28s} {'ok' if hasattr(_mw, name) else 'MISSING on this build'}")

# ToolErrorMiddleware is newer than the rest. If it says MISSING, Step 7 shows the
# hand-written equivalent, which works on every 1.x build.

In [ ]:
import inspect, json, re, time, textwrap
from collections.abc import Callable

from langchain.tools import tool
from langchain.agents import create_agent
from langchain.agents.middleware import (
    HumanInTheLoopMiddleware, PIIMiddleware, SummarizationMiddleware,
    ToolRetryMiddleware, ToolCallLimitMiddleware, ModelCallLimitMiddleware,
    wrap_tool_call, before_model, AgentState,
)
from langchain.messages import ToolMessage
from langchain.tools.tool_node import ToolCallRequest
from langchain_core.messages import AIMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command

print("imports ok")

# If langchain.messages does not resolve on your build, langchain_core.messages has the same classes.

---
# 3. The map

Read this once before writing anything. Everything in the notebook is a box on this picture.

```text
                         INBOUND EMAIL
                              |
                              v
                   +----------------------+
                   |  LCEL TRIAGE CHAIN   |   prompt -> model -> json parser
                   +----------------------+   fixed path, no tools, cheap
                              |
                      needs reasoning?
                       /            \
                     no              yes
                     |                |
                     v                v
              direct lookup   +--------------------------------------+
              no LLM loop     |             AGENT LOOP               |
                              |                                      |
                              |  [before model]                      |
                              |     1  PII redact input              |
                              |     2  compact long history          |
                              |              |                       |
                              |              v                       |
                              |         MODEL CALL                   |
                              |              |                       |
                              |              v                       |
                              |  [after model]                       |
                              |     3  approval gate  --- pause ---> | ---> HUMAN
                              |              |                       | <--- decision
                              |              v                       |
                              |  [around each tool call]             |
                              |     4  guard      short circuit      |
                              |     5  safety net catch              |
                              |     6  retry      re-run             |
                              |              |                       |
                              |              v                       |
                              |          TOOL RUNS  -----------------|---> carrier API
                              |              |                       |     booking system
                              |              +--> back to model      |
                              +--------------------------------------+
                                            |     ^
                                            v     |
                                   CHECKPOINTER (thread_id = AWB)
                                            |
                                            v
                                     REPLY TO DESK AGENT
```

## Two planes

| Plane | What lives there | In the picture |
|---|---|---|
| **Execution** | Work that changes something | Model call, tools |
| **Control** | Rules about what is allowed | Triage, gate, guard, safety net, retry, PII, compaction, checkpointer |

Almost everything you add today is control plane. That ratio is normal, and it is the difference
between a demo and a system.

## The chain of 16 steps

| Step | What you add | What it leaves broken |
|---|---|---|
| 1 | A model call | A model with no hands |
| 2 | LCEL triage chain | Real cases still need tools |
| 3 | Five tools | Nothing calls them |
| 4 | `create_agent` loop | A typo ends the run with a traceback |
| 5 | Error split rule | The carrier API times out and the case is lost |
| 6 | Retry, scoped | Only predicted failures are covered |
| 7 | Safety net and guard | Nothing crashes, so the agent completes a write unsupervised |
| 8 | Checkpointer memory | It remembers the case and still writes unsupervised |
| 9 | Approval gate | Approval screens and traces now carry shipper identifiers |
| 10 | PII layers | Long threads still grow without limit |
| 11 | Compaction | Compaction can drop facts later turns need |
| 12 | Limits | You cannot see what the agent is doing |
| 13 | Audit trace | You do not know what order any of this runs in |
| 14 | Order rules | Now assemble it |
| 15 | The full agent | It works, until persistence fails |
| 16 | Fail-closed degradation | Nothing. This is the one that matters |

---
# Step 1: One model call

```text
   text  -->  [ MODEL ]  -->  text
```

That is all a chat model does. It has no memory, no tools and no ability to act.
Everything else in this notebook is scaffolding built around this one box.

**Offline mode.** If `MODE = "offline"` you get `ScriptedModel`, a deterministic stand-in that
follows fixed rules instead of reasoning. It is not a toy: a test double like this is how you write
agent tests that pass or fail on logic instead of on model mood. LangChain ships a similar idea as
`LLMToolEmulator`.

In [ ]:
from langchain_core.language_models.chat_models import BaseChatModel
from langchain_core.outputs import ChatGeneration, ChatResult

class ScriptedModel(BaseChatModel):
    """Deterministic stand-in for a real model.

    Rules, in order:
      1. Asked to summarize   -> return a short plain-text summary
      2. Asked to classify    -> return triage JSON
      3. Otherwise            -> walk the cargo workflow one tool at a time
    """
    allowed: list = []

    @property
    def _llm_type(self) -> str:
        return "scripted"

    def bind_tools(self, tools, **kwargs):
        names = []
        for t in tools:
            n = getattr(t, "name", None) or (t.get("name") if isinstance(t, dict) else None)
            if n:
                names.append(n)
        object.__setattr__(self, "allowed", names)
        return self

    def _generate(self, messages, stop=None, run_manager=None, **kwargs) -> ChatResult:
        blob = "\n".join(str(getattr(m, "content", "")) for m in messages)
        first = str(getattr(messages[0], "content", "")) if messages else ""

        def out(msg):
            return ChatResult(generations=[ChatGeneration(message=msg)])

        # 1. summarization request
        if "<messages>" in blob or "Context Extraction" in blob:
            return out(AIMessage(content="SUMMARY: cargo disruption case in progress, "
                                         "shipment identified, alternates reviewed."))

        # 2. triage request
        if "classify" in first.lower():
            m = re.search(r"\d{3}-\d{8}", blob)
            disrupted = any(w in blob.lower() for w in ("cancel", "delay", "disrupt", "aog"))
            return out(AIMessage(content=json.dumps({
                "case_type": "disruption" if disrupted else "status_query",
                "awb": m.group(0) if m else None,
                "urgency": "high" if disrupted else "normal",
            })))

        # 3. agent workflow
        tool_msgs = [m for m in messages if getattr(m, "type", "") == "tool"]
        done = {getattr(m, "name", None) for m in tool_msgs}
        said = " ".join(str(getattr(m, "content", "")) for m in messages
                        if getattr(m, "type", "") == "human").lower()
        seen = said + " " + " ".join(str(getattr(m, "content", "")) for m in tool_msgs)

        # 3a. a tool reported a condition that cannot be worked around
        stop_words = ("NOT_FOUND", "TOOL_FAILED", "BLOCKED", "NO_OPTIONS", "ALREADY_REBOOKED")
        if any(str(getattr(m, "content", "")).startswith(stop_words) for m in tool_msgs):
            return out(AIMessage(content="A tool reported a condition I cannot work around. "
                                         "Passing this to the desk agent."))

        # 3b. diagnostic probe, so error handling can be demonstrated
        if "boom" in self.allowed and "boom" not in done:
            return out(AIMessage(content="", tool_calls=[
                {"name": "boom", "args": {"x": "probe"}, "id": "call_boom"}]))

        # 3c. asked for something no tool covers
        if any(w in said for w in ("truck", "road transport", "by sea")):
            return out(AIMessage(content="I have no tool for road or sea transport. "
                                         "That request has to go to the desk agent."))

        # 3d. the write was rejected by a human, so do not claim it happened
        for m_ in tool_msgs:
            if getattr(m_, "name", "") == "rebook_shipment" and not str(m_.content).startswith("REBOOKED"):
                return out(AIMessage(content="The rebooking was not carried out. "
                                             "Reviewer feedback: " + str(m_.content)[:90]))

        found = re.search(r"\d{3}-\d{8}", seen)
        if found is None and not done:
            return out(AIMessage(content="I have no case context on this thread. "
                                         "Give me an AWB and I will start."))
        awb = found.group(0) if found else CASE_AWB
        book = globals().get("SHIPMENTS", {}).get(awb, {})
        flight = book.get("flight", "MA-217")
        dep = book.get("dep_date", "2026-08-06")
        origin = book.get("origin", "BLR")
        dest = book.get("destination", "AMS")

        disrupted = any(w in said for w in
                        ("cancel", "delay", "disrupt", "aog", "advise", "rebook", "reroute"))

        plan = [("find_shipment", {"awb": awb}),
                ("get_flight_status", {"flight_no": flight, "dep_date": dep})]
        if disrupted:
            plan += [
                ("find_alternate_flights", {"origin": origin, "destination": dest,
                                            "earliest_date": dep}),
                ("rebook_shipment", {"awb": awb, "flight_no": "MA-219", "dep_date": dep}),
            ]
        for i, (name, args) in enumerate(plan):
            if name in self.allowed and name not in done:
                return out(AIMessage(content="", tool_calls=[
                    {"name": name, "args": args, "id": f"call_{i}"}]))

        if not disrupted:
            return out(AIMessage(content=f"Shipment {awb} located and its flight status checked. "
                                         "No disruption action proposed."))
        return out(AIMessage(content="Rebooked onto MA-219, cold chain confirmed. "
                                     "Shipper informed of the new departure."))

print("ScriptedModel defined")

In [ ]:
def build_model():
    """One place that decides where the model comes from."""
    if MODE == "bedrock":
        from langchain_aws import ChatBedrockConverse
        return ChatBedrockConverse(
            model=MODEL_ID,
            region_name=REGION,
            temperature=TEMPERATURE,     # temperature XOR top_p on Claude 4.x
            max_tokens=MAX_TOKENS,
        )
    return ScriptedModel()

llm = build_model()
print("MODE =", MODE, "| model =", type(llm).__name__)

reply = llm.invoke([{"role": "user", "content": "One line: what is an air waybill?"}])
print("\n", reply.content)

### What to notice

| Observation | Why it matters |
|---|---|
| The call took a list of messages, not a string | Everything upstream eventually becomes a message list |
| Nothing was remembered | A second call would start blank. Memory is Step 8 |
| No tool could have been used | The model can only produce text right now |

**Production fact.** `temperature` and `top_p` are mutually exclusive on Claude 4.x through Bedrock.
Setting both raises a validation error. And the `us.` prefix on the model id is a cross-region
inference profile, not decoration: the bare model id raises `ValidationException`.

---
# Step 2: Route before you reason

```text
   email --> [ PROMPT ] --> [ MODEL ] --> [ JSON PARSER ] --> {case_type, awb, urgency}
                                                                    |
                                                        +-----------+-----------+
                                                        |           |           |
                                                   status_query  disruption   other
                                                        |           |           |
                                                        v           v           v
                                                  direct read    agent      human queue
```

An LCEL chain is a fixed pipeline. Same shape every call, no tools, no loop, cheap, testable.
That makes it the right thing to put **in front of** an agent.

The pipe reads left to right as data flow. Text into a prompt, prompt into a model, model output
into a parser.

**Why this is the first thing you build in industry:** an agent loop is the most expensive way to
answer a question. Most inbound traffic does not need one. Decide before you pay.

In [ ]:
TRIAGE_PROMPT = ChatPromptTemplate.from_messages([
    ("system",
     "You classify inbound air cargo desk emails. "
     "Reply with ONLY a JSON object, no prose, no code fences. "
     "Keys: case_type, awb, urgency. "
     "case_type is one of: status_query, disruption, claim, other. "
     "awb is the air waybill as 3 digits, dash, 8 digits, or null. "
     "urgency is one of: low, normal, high."),
    ("human", "{email}"),
])

triage_chain = TRIAGE_PROMPT | build_model() | JsonOutputParser()

def route(email_text: str) -> tuple:
    t = triage_chain.invoke({"email": email_text})
    if t["case_type"] == "status_query" and t["awb"]:
        return "fast_path", t
    if t["case_type"] in ("disruption", "claim"):
        return "agent", t
    return "human_queue", t

for sample in ["Status of AWB 160-44120087 please?", CASE_TEXT]:
    lane, detail = route(sample)
    print(f"{lane:12s} <- {detail}")

### What to notice

- The chain is three objects joined by `|`. No class, no loop, no state.
- `JsonOutputParser` turns text into a dict, so `route()` is ordinary Python from there.
- Reverse the order and it breaks. A parser has nothing to parse before the model runs.

**Production fact.** This chain is the cheapest test surface you own. Point it at 200 historical
emails tonight and you will know its accuracy before a single agent runs in production. You cannot
say that about the agent loop.

**Where it fails.** If almost nothing routes to `fast_path`, the router is decoration. Measure the
split before you keep it.

---
# Step 3: Tools

```text
   +--------------------------------------------------+
   |  @tool                                           |
   |    name       <- the function name               |
   |    description<- the DOCSTRING                   |  <-- the model reads this
   |    args schema<- the TYPE HINTS                  |  <-- and this
   +--------------------------------------------------+
                          |
                   model picks a tool
                   and fills the args
```

A tool is a function plus a contract. The docstring is not a comment. It is the API documentation
the model reads at selection time, and it is the single biggest lever on whether the right tool
gets called.

**Write docstrings for the model, not for your team.** State what it returns, and when to call it.

In [ ]:
class CarrierTimeout(Exception):
    """Carrier status API did not answer in time. Transient."""

class CarrierUnavailable(Exception):
    """Carrier status API returned 5xx. Transient."""

SHIPMENTS = {
    "160-45872910": {
        "origin": "BLR", "destination": "AMS", "pieces": 12, "gross_kg": 480,
        "flight": "MA-217", "dep_date": "2026-08-06",
        "commodity": "pharma, 2 to 8 C, 30h reserve",
        "shipper_email": "ops@kavery-exports.example",
        "shipper_phone": "+91 98450 11223",
    },
    "160-44120087": {
        "origin": "BLR", "destination": "FRA", "pieces": 3, "gross_kg": 91,
        "flight": "MA-204", "dep_date": "2026-08-06",
        "commodity": "machine spares",
        "shipper_email": "logistics@tarun-industrial.example",
        "shipper_phone": "+91 80471 55010",
    },
}
_REBOOKED = {}
_status_calls = {"n": 0}


@tool
def find_shipment(awb: str) -> str:
    """Look up one shipment by Air Waybill number, format 3 digits dash 8 digits.

    Returns origin, destination, pieces, gross weight, booked flight, departure date,
    commodity and shipper contact. Call this first whenever the user names an AWB.
    """
    r = SHIPMENTS.get(awb.strip())
    if r is None:
        return f"NOT_FOUND: no shipment matches AWB {awb}. Ask the user to re-check the number."
    return (f"AWB {awb}: {r['origin']} to {r['destination']}, {r['pieces']} pcs / {r['gross_kg']} kg, "
            f"booked {r['flight']} dep {r['dep_date']}, commodity {r['commodity']}, "
            f"shipper contact {r['shipper_email']} / {r['shipper_phone']}")


@tool
def get_flight_status(flight_no: str, dep_date: str) -> str:
    """Get live status for a flight such as MA-217 on a date in YYYY-MM-DD format.

    Returns ON_TIME, DELAYED or CANCELLED with a reason where the carrier gives one.
    """
    _status_calls["n"] += 1
    if _status_calls["n"] < CARRIER_FLAKY_UNTIL:
        raise CarrierTimeout(f"carrier API no response in 5s for {flight_no}")
    if (flight_no, dep_date) == ("MA-217", "2026-08-06"):
        return "CANCELLED: MA-217 on 2026-08-06 cancelled, aircraft AOG at BLR."
    return f"ON_TIME: {flight_no} on {dep_date}, no disruption recorded."


@tool
def find_alternate_flights(origin: str, destination: str, earliest_date: str) -> str:
    """Find flights with available cargo capacity between two stations from a date.

    Returns flight numbers with departure time, free capacity and cold chain capability.
    Returns NO_OPTIONS when nothing is available in the next 72 hours.
    """
    if (origin, destination) == ("BLR", "AMS"):
        return ("MA-219 dep 2026-08-06 21:40, 900 kg free, cold chain YES; "
                "PT-881 dep 2026-08-07 04:15, 1400 kg free, cold chain NO; "
                "MA-217 dep 2026-08-07 09:00, 600 kg free, cold chain YES")
    return "NO_OPTIONS: no capacity found on this sector in the next 72 hours."


@tool
def rebook_shipment(awb: str, flight_no: str, dep_date: str) -> str:
    """WRITE ACTION. Move a shipment onto a different flight and reissue the booking.

    Changes the live booking and triggers downstream handling instructions. Only call it
    after confirming the alternate has capacity and the right cold chain capability.
    """
    if awb in _REBOOKED:
        return f"ALREADY_REBOOKED: {awb} is already on {_REBOOKED[awb]}. No change made."
    _REBOOKED[awb] = flight_no
    return f"REBOOKED: {awb} moved to {flight_no} dep {dep_date}. Booking ref RB-{abs(hash(awb)) % 100000}."


@tool
def notify_customer(contact: str, message: str) -> str:
    """WRITE ACTION. Send a message to the shipper contact for a shipment.

    contact is an email address or phone number. message is the text to send.
    """
    return f"SENT to {contact}: {message[:80]}"


READ_TOOLS  = [find_shipment, get_flight_status, find_alternate_flights]
WRITE_TOOLS = [rebook_shipment, notify_customer]
ALL_TOOLS   = READ_TOOLS + WRITE_TOOLS

print(f"{len(ALL_TOOLS)} tools defined")

In [ ]:
# See exactly what the model sees. Do this whenever a model picks the wrong tool:
# nine times out of ten the description is the bug, not the model.
for t in ALL_TOOLS:
    print(f"{t.name}")
    print(f"   args : {list(t.args.keys())}")
    print(f"   desc : {t.description.splitlines()[0]}")

### The idempotency line, and why it is there

```python
if awb in _REBOOKED:
    return f"ALREADY_REBOOKED: ..."
```

Step 6 adds retries. Retries re-run calls. A human on a slow connection can also approve twice.
Without that check, both produce a duplicate booking on live freight.

**Rule:** every write tool needs an idempotency check before it needs anything else.

**Production fact.** In a real system the key is not a module dict. It is a server-side
idempotency key on the booking API, so a retry from any client is deduplicated at the source of
truth rather than in your agent process.

---
# Step 4: The agent loop

```text
      user message
           |
           v
      +---------+   no tool calls
      |  MODEL  | ------------------> final answer
      +---------+
           | tool calls
           v
      +---------+
      |  TOOLS  |
      +---------+
           |
           +--> results appended to messages, back to MODEL
```

`create_agent` is that loop. The model is called, and if the reply contains tool calls the tools
run and their results are appended as messages. Repeat until the model replies without tool calls.

Nothing more. Everything you add after this is a rule about *when* the loop may proceed.

In [ ]:
SYSTEM_PROMPT = """You are FreightDesk, an assistant to air cargo exception desk agents.

Working rules:
- Establish the shipment with find_shipment before advising anything.
- Check live flight status before assuming a disruption is real.
- For temperature controlled commodities, only propose alternates with cold chain capability.
- Never claim a rebooking has happened unless a tool result confirms it.
- rebook_shipment and notify_customer act on live systems. Propose them, do not narrate them as done.
- When a tool returns NOT_FOUND, NO_OPTIONS or TOOL_FAILED, say so plainly and stop.
"""

def show(result, title="TRACE"):
    """Print a message trace the way you would want it in a log."""
    print(f"--- {title} " + "-" * (56 - len(title)))
    for m in result["messages"]:
        kind = m.type.upper()
        if getattr(m, "tool_calls", None):
            for tc in m.tool_calls:
                print(f"  {kind:9s} calls {tc['name']}({tc['args']})")
        elif m.type == "tool":
            print(f"  {kind:9s} {m.name}: {str(m.content)[:88]}")
        elif str(m.content).strip():
            print(f"  {kind:9s} {str(m.content)[:88]}")
    if "__interrupt__" in result:
        print("  PAUSED    waiting for a human decision")
    print()

_status_calls["n"] = CARRIER_FLAKY_UNTIL     # skip the flaky simulation for this step only
_REBOOKED.clear()

basic_agent = create_agent(model=build_model(), tools=READ_TOOLS, system_prompt=SYSTEM_PROMPT)
res = basic_agent.invoke({"messages": [{"role": "user", "content": CASE_TEXT}]})
show(res, "STEP 4 read-only agent")

### What to notice

- Three tool calls happened without you writing a single `if`. The model chose the order.
- The tool results are ordinary messages. There is no hidden channel.
- The loop ended when the model replied with text instead of a tool call.

**Production fact.** This loop has no ceiling by default. A model that keeps calling tools keeps
costing money. Step 12 adds the ceiling, and you should never ship without one.

---
# Step 5: The error split, the most important rule in the notebook

```text
   something went wrong inside a tool
                  |
                  v
      is it a normal business outcome?
       (no such AWB, no capacity, already booked)
              /             \
            yes              no
             |                |
             v                v
      RETURN a string    would the same call
      the model reads    succeed in 2 seconds?
             |             /            \
             |           yes             no
             v            |               |
      model replans       v               v
                    RAISE a typed    RAISE
                    exception        anything
                          |               |
                          v               v
                    retry re-runs    safety net converts
                    the call         it to a message
```

`create_agent` **does not catch tool exceptions for you.** The default handler returns the message
for `ToolInvocationError` and re-raises everything else. An unhandled `KeyError` in a lookup ends
the run.

So the choice of raise versus return is architecture, not style.

In [ ]:
@tool
def boom(x: str) -> str:
    """Diagnostic tool that always fails. Used to demonstrate error handling."""
    raise KeyError("internal_field_missing")

# A. no protection
fragile = create_agent(model=build_model(), tools=[boom], system_prompt="Call boom once.")
try:
    fragile.invoke({"messages": [{"role": "user", "content": "run the probe"}]})
    print("A: run completed")
except Exception as exc:
    print(f"A: run HALTED with {type(exc).__name__}: {exc}")

# B. an expected outcome returned as data
print("\nB:", find_shipment.invoke({"awb": "160-99999999"}))

### The rule, in one table

| Situation | What to do | Why |
|---|---|---|
| No such AWB, no capacity, already rebooked | `return "NOT_FOUND: ..."` | The model can act on it. The case continues |
| Timeout, 5xx, throttle | `raise CarrierTimeout(...)` | Only an exception is visible to the retry layer |
| Bug, bad config, auth failure | let it raise | The safety net in Step 7 turns it into a readable message |

Both wrong directions cost you. Returning a timeout as text hides it from retry, so nothing
retries and the model learns to give up. Raising "no such AWB" ends a case over a typo.

**Production fact.** Never put a raw exception message into a tool result. It carries hostnames,
connection strings and file paths straight into a model prompt and into your traces. Name the
exception type instead.

---
# Step 6: Retry, scoped and typed

```text
   tool call --> [ RETRY LAYER ] --> tool runs
                        ^                |
                        |            raises
                        |                |
                        |                v
                        |     type listed in retry_on?
                        |         /              \
                        |       no                yes
                        |        |                 |
                        |        v                 v
                        |   propagate now     attempts left?
                        |                      /         \
                        +--------------------yes          no
                          wait, then re-run               |
                                                          v
                                              on_failure = "continue"
                                              -> error message to model
                                              on_failure = "error"
                                              -> re-raise, run stops
```

Retry is scoped, typed and bounded, or it is a liability.

| Setting | Wrong value | What it costs |
|---|---|---|
| `retry_on` | `(Exception,)`, which is the default | Re-runs your own bugs three times with backoff, then fails anyway |
| `tools` | left unset | Retries write tools, and double-books freight |
| `on_failure` | `"error"` when you meant continue | Stops the run instead of letting the model recover |

In [ ]:
transient_retry = ToolRetryMiddleware(
    max_retries=RETRY_MAX_ATTEMPTS,
    initial_delay=RETRY_INITIAL_DELAY,
    backoff_factor=RETRY_BACKOFF,
    jitter=True,
    tools=RETRY_TOOLS,                                    # scope, never global
    retry_on=(CarrierTimeout, CarrierUnavailable),        # only what time can fix
    on_failure="continue",                                # let the model see an exhausted failure
)

_status_calls["n"] = 0        # re-arm the flaky carrier simulation
retry_agent = create_agent(model=build_model(), tools=READ_TOOLS,
                           system_prompt=SYSTEM_PROMPT, middleware=[transient_retry])
res = retry_agent.invoke({"messages": [{"role": "user", "content": CASE_TEXT}]})
show(res, "STEP 6 with retry")

seen = sum(1 for m in res["messages"] if m.type == "tool" and m.name == "get_flight_status")
print(f"tool actually ran     : {_status_calls['n']} times")
print(f"results the model saw : {seen}")
print("The retries never reached the model. That is the point.")

### What to notice

The counter and the trace disagree, on purpose. The tool ran three times, the model saw one
result. Retry absorbed the noise instead of teaching the model that the carrier API is unreliable.

**Production fact.** Turn `jitter` on whenever more than one process can retry the same dependency.
Without it, every client backs off on the same schedule and hits the recovering service in a
synchronized wave. That is the thundering herd, and it turns a blip into an outage.

---
# Step 7: The safety net and the guard

```text
   model emits a tool call
             |
             v
      [ GUARD ]  args look wrong?
             |         \
             |          +--> return a message NOW, handler never called
             v                (short circuit: the tool never runs)
      [ SAFETY NET ]
             |
             v
      [ RETRY ]
             |
             v
         TOOL RUNS
             |
       raises something
       nobody predicted
             |
             v
      caught by SAFETY NET --> ToolMessage(status="error") --> model replans
```

One hook, `wrap_tool_call`, does two different jobs.

| Move | How | Use for |
|---|---|---|
| **Catch** | wrap `handler(request)` in `try` | Turning crashes into readable messages |
| **Short circuit** | return without calling `handler` | Blocking a call from ever running |

A guard beats a prompt instruction. A prompt asks the model not to do something. A guard makes it
impossible, and it does not degrade as context grows.

In [ ]:
@wrap_tool_call
def tool_safety_net(request: ToolCallRequest,
                    handler: Callable[[ToolCallRequest], ToolMessage | Command]):
    """Turn an unexpected tool exception into something the model can read."""
    try:
        return handler(request)
    except Exception as exc:
        return ToolMessage(
            content=(f"TOOL_FAILED: {request.tool_call['name']} raised "
                     f"{type(exc).__name__}. Report this to the user, do not retry."),
            tool_call_id=request.tool_call["id"],   # MUST match, or history is malformed
            name=request.tool_call["name"],         # so traces show which tool failed
            status="error",
        )


@wrap_tool_call
def write_guard(request: ToolCallRequest,
                handler: Callable[[ToolCallRequest], ToolMessage | Command]):
    """Refuse a rebooking that does not name a shipment."""
    call = request.tool_call
    if call["name"] == "rebook_shipment" and not call["args"].get("awb"):
        return ToolMessage(content="BLOCKED: rebook_shipment requires an awb argument.",
                           tool_call_id=call["id"], name=call["name"], status="error")
    return handler(request)


# same failing tool as Step 5, now with the net
safe = create_agent(model=build_model(), tools=[boom], system_prompt="Call boom once.",
                    middleware=[tool_safety_net])
res = safe.invoke({"messages": [{"role": "user", "content": "run the probe"}]})
show(res, "STEP 7 same crash, contained")

### Three details in the safety net that all matter

| Line | Why |
|---|---|
| `tool_call_id=request.tool_call["id"]` | Must match the call the model made. Otherwise history carries an unanswered tool call and the next model call fails on a different error than the one you were fixing |
| `status="error"` | Marks it as a failure rather than a result |
| `type(exc).__name__`, not `str(exc)` | Keeps internal detail out of the prompt and the trace |

**Never use `yield` inside a `wrap_tool_call` function.** It becomes a generator and raises
`NotImplementedError`.

**Production fact.** `ToolErrorMiddleware` does this job on recent builds. If your preflight said
MISSING, the hand-written version above works on every 1.x build and is worth understanding anyway,
because it is also where you put rate limiting, per-tenant authorization and audit hooks.

---
# Step 8: Memory

```text
   turn 1 --> [ AGENT ] --> reply
                  |
                  v
          CHECKPOINTER  saves state under thread_id
                  |
   turn 2 --> [ AGENT ] <-+  loads the same thread_id back
                  |
                  v
              reply that knows about turn 1
```

A checkpointer saves the state of a run so the next call on the same `thread_id` continues instead
of starting over.

**One case equals one thread.** Make `thread_id` the AWB and the technical memory boundary becomes
the business boundary. Two cases sharing a thread means shipment A's context steers shipment B's
reasoning, and that bug survives QA.

In [ ]:
checkpointer = InMemorySaver()
mem_agent = create_agent(model=build_model(), tools=READ_TOOLS,
                         system_prompt=SYSTEM_PROMPT,
                         middleware=[tool_safety_net, transient_retry],
                         checkpointer=checkpointer)

case_config  = {"configurable": {"thread_id": CASE_AWB}}
other_config = {"configurable": {"thread_id": "some-other-case"}}

_status_calls["n"] = CARRIER_FLAKY_UNTIL
mem_agent.invoke({"messages": [{"role": "user", "content": CASE_TEXT}]}, config=case_config)

same  = mem_agent.invoke({"messages": [{"role": "user", "content": "what did we just do?"}]},
                         config=case_config)
other = mem_agent.invoke({"messages": [{"role": "user", "content": "what did we just do?"}]},
                         config=other_config)

print("same thread   :", same["messages"][-1].content[:100])
print("other thread  :", other["messages"][-1].content[:100])
print("\nmessages held on the case thread :", len(same["messages"]))
print("messages held on the other thread:", len(other["messages"]))

### What to notice

The two threads gave different answers to the same question. That is isolation, and it is the
whole reason `thread_id` exists.

Any other key name in `configurable` is **silently ignored**. You get a working agent with no
memory and no error message, which is the worst combination available.

**Production fact.** `InMemorySaver` dies with the process. Production uses a durable checkpointer
such as Postgres or Mongo. That matters more than it sounds, and Step 16 is entirely about why.

---
# Step 9: The approval gate

```text
   MODEL decides to call rebook_shipment
             |
             v
   [ after model ]  gate reads the proposed call
             |
      needs a human?
        /        \
      no          yes
      |            |
      v            v
   tool runs   state saved to checkpointer
               run pauses and returns
                     |
                     v
              human sees: name, args, allowed decisions
                     |
                     v
              Command(resume={"decisions": [...]})
                     |
                     v
              approved -> tool runs with those args
              rejected -> tool skipped, feedback goes to the model
```

The gate runs **after the model responds and before the tools run**. That timing is the design.
The model has decided what it wants to do and nothing has happened yet. It is the only moment
where approve, edit and reject are all still meaningful.

A gate placed after the side effect is not a control. It is a notification.

**And it stands on the checkpointer.** Pausing means writing state somewhere durable. No
checkpointer, no pause. Remember that at Step 16.

| Decision | What happens | Use for |
|---|---|---|
| `approve` | Tool runs with the original args | The proposal is right |
| `edit` | Tool runs with your args | Small, enumerable changes such as a different flight |
| `reject` | Tool skipped, your message goes back as feedback | Denying an action |
| `respond` | Your message is returned as a **successful** tool result | Only for tools that exist to ask a human something |

Never deny a write with `respond`. It tells the model the write succeeded.

In [ ]:
approval_gate = HumanInTheLoopMiddleware(
    interrupt_on={
        **{name: {"allowed_decisions": decisions} for name, decisions in GATE_ON.items()},
        **{t.name: False for t in READ_TOOLS},        # reads run freely, on purpose
    },
    description_prefix=GATE_PREFIX,
)

_status_calls["n"] = CARRIER_FLAKY_UNTIL
_REBOOKED.clear()
gated = create_agent(model=build_model(), tools=ALL_TOOLS, system_prompt=SYSTEM_PROMPT,
                     middleware=[write_guard, tool_safety_net, transient_retry, approval_gate],
                     checkpointer=InMemorySaver())
cfg = {"configurable": {"thread_id": "gate-demo"}}

paused = gated.invoke({"messages": [{"role": "user", "content": CASE_TEXT}]}, config=cfg)
show(paused, "STEP 9 paused before the write")

if "__interrupt__" in paused:
    payload = paused["__interrupt__"][0].value
    for req in payload["action_requests"]:
        print("PENDING :", req["name"], req["args"])          # note the key is args, not arguments
    for rc in payload["review_configs"]:
        print("ALLOWED :", rc["action_name"], rc["allowed_decisions"])

In [ ]:
# The desk agent approves. One decision per pending action, in the same order.
final = gated.invoke(Command(resume={"decisions": [{"type": "approve"}]}), config=cfg)
show(final, "STEP 9 after approve")
print("FINAL:", final["messages"][-1].content[:160])

In [ ]:
# Same case on a fresh thread, this time rejected with feedback the model can use.
_status_calls["n"] = CARRIER_FLAKY_UNTIL
_REBOOKED.clear()
cfg2 = {"configurable": {"thread_id": "gate-demo-reject"}}

gated.invoke({"messages": [{"role": "user", "content": CASE_TEXT}]}, config=cfg2)
rejected = gated.invoke(
    Command(resume={"decisions": [{
        "type": "reject",
        "message": "PT-881 has no cold chain. Do not use it. Re-check MA-219 capacity first.",
    }]}),
    config=cfg2,
)
show(rejected, "STEP 9 after reject")
print("bookings actually made:", _REBOOKED)

### What to notice

| Observation | Why it matters |
|---|---|
| Reads were never gated | A desk agent approving lookups forty times a morning stops reading arguments. Gate side effects only |
| The pending action showed args, not prose | Approvals must show the exact arguments, or the record is theatre |
| After reject, `_REBOOKED` stayed empty | The write really did not happen. Verify this, do not assume it |

**Production fact.** The interrupt payload key is `args`, not `arguments`, whatever some examples
show. Print `result["__interrupt__"][0].value` once on your build and read the shape rather than
copying it.

**Where it fails.** Approval fatigue. 300 rebookings a shift and the gate becomes a click, which
gives you an approval record nobody read. The real fix is a conditional gate: interrupt only above
a weight, value or commodity threshold. That needs the `when` predicate, available on recent builds.

---
# Step 10: Privacy, three doors not one

```text
   door 1                 door 2                  door 3
   user input             tool results            model output
      |                      |                        |
      v                      v                        v
  apply_to_input=True   apply_to_tool_results=True  apply_to_output=True
      |                      |                        |
      +----------+-----------+------------------------+
                 |
                 v
             MODEL CONTEXT, LOGS, TRACES
```

Most teams wire redaction on user input, tick the compliance box, and miss the door the data
actually comes through.

In this design the shipper email and phone never appear in user input. They come out of
`find_shipment`. Guard only door 1 and you have a clean compliance answer that is false.

| Strategy | Model sees | Use when |
|---|---|---|
| `redact` | `[REDACTED_EMAIL]` | The value is irrelevant to the reasoning |
| `mask` | `****1223` | The tail matters for confirmation |
| `hash` | stable hash | You need to match the same entity across turns |
| `block` | exception raised | The value must never enter the system |

Built-in detectors: `email`, `credit_card`, `ip`, `mac_address`, `url`. Anything else needs your own
`detector=`, or the constructor raises.

In [ ]:
pii_layers = [
    PIIMiddleware("email", strategy=PII_EMAIL_STRATEGY,
                  apply_to_input=PII_ON_INPUT,
                  apply_to_tool_results=PII_ON_TOOL_RESULTS,
                  apply_to_output=PII_ON_OUTPUT),
    PIIMiddleware("phone_number", detector=PII_PHONE_REGEX, strategy=PII_PHONE_STRATEGY,
                  apply_to_input=PII_ON_INPUT,
                  apply_to_tool_results=PII_ON_TOOL_RESULTS),
]

def contact_demo(layers, label):
    a = create_agent(model=build_model(), tools=[find_shipment],
                     system_prompt=SYSTEM_PROMPT, middleware=layers)
    out = a.invoke({"messages": [{"role": "user",
        "content": f"check awb {CASE_AWB}, reply to ops@kavery-exports.example or +91 98450 11223"}]})
    print(f"[{label}]")
    print("   user  :", out["messages"][0].content)
    for m in out["messages"]:
        if m.type == "tool":
            print("   tool  :", str(m.content)[:110])
    print()

contact_demo([], "no layers")
contact_demo([pii_layers[0]], "email only, input + tool results")
contact_demo(pii_layers, "email + phone")

In [ ]:
# The trap nobody warns you about: a detector that is too greedy.
# This domain is made of numbers. Test yours against real business identifiers.
candidates = {
    "greedy": r"\+?\d[\d\s().-]{7,}\d",
    "config (leading + required)": PII_PHONE_REGEX,
}
probe = ["+91 98450 11223", "160-45872910", "2026-08-06", "MA-217", "480 kg", "9845011223"]

for label, pattern in candidates.items():
    print(f"\n{label}: {pattern}")
    for s in probe:
        hit = re.findall(pattern, s)
        verdict = "MATCH" if hit else "-"
        note = ""
        if hit and s != "+91 98450 11223":
            note = "   <-- FALSE POSITIVE, this is business data"
        if not hit and s == "9845011223":
            note = "   <-- false negative, a real local-format number"
        print(f"   {s:20s} {verdict:6s}{note}")

### What the greedy detector actually does to you

Run the agent with it and the tool result comes back as:

```text
AWB ****2910: BLR to AMS, 12 pcs / 480 kg, booked MA-217 dep ****8-06
```

The AWB and the departure date are gone. The agent is now reasoning about a shipment it cannot
name, and every downstream tool call carries masked identifiers.

**Neither pattern is clean.** Requiring a leading `+` protects your identifiers and misses local
format numbers. That tradeoff is the design decision, and no setting removes it. Pick deliberately,
and write a fixture of real strings to score it.

**Production fact.** Redaction breaks the tool that needs the data. `notify_customer` needs a real
address. The correct fix is not to disable redaction on that path, it is to pass a reference such as
the AWB and resolve the contact inside the tool, outside the model's context entirely.

---
# Step 11: Compaction

```text
   history grows
        |
        v
   trigger met?  ----no----> pass through untouched
        |
       yes
        |
        v
   split: [ older messages ] + [ recent, per keep ]
              |
              v
        summarize the older part
              |
              v
   summary + recent messages --> MODEL
```

A long case thread carries every tool result from turn 1. Eventually it stops fitting, and it costs
you on every single call before it does.

`trigger` and `keep` are both `(unit, value)` tuples where the unit is `fraction`, `tokens` or
`messages`.

**The default `trigger` is `None`, and `None` means summarization never fires.** A
`SummarizationMiddleware` created without a trigger is a no-op that looks like a control.

In [ ]:
print("signature:", inspect.signature(SummarizationMiddleware.__init__))
print()

default_mw = SummarizationMiddleware(model=build_model())
print("default trigger :", default_mw.trigger, "  <-- None means it never fires")
print("default keep    :", default_mw.keep)

compaction = SummarizationMiddleware(
    model=build_model(),
    trigger=("messages", COMPACT_TRIGGER_MESSAGES),
    keep=("messages", COMPACT_KEEP_MESSAGES),
)
print("\nconfigured trigger:", compaction.trigger)
print("configured keep   :", compaction.keep)

In [ ]:
# Watch the message count stop growing linearly.
chatty = create_agent(model=build_model(), tools=[], system_prompt="Reply in one short line.",
                      middleware=[compaction], checkpointer=InMemorySaver())
ccfg = {"configurable": {"thread_id": "long-case"}}

for i in range(1, 13):
    out = chatty.invoke({"messages": [{"role": "user", "content": f"note {i}"}]}, config=ccfg)
    if i % 3 == 0:
        print(f"after turn {i:2d}: {len(out['messages']):3d} messages in state")

### The cost you just accepted

Compaction turns older tool results into prose. The AWB, the cold chain flag and any completed
write are facts later turns depend on. If the summary drops the cold chain flag, turn 22 proposes
the freighter with no cold chain.

**Production fact.** Anything a later decision depends on belongs in structured state, not in a
prose summary. Two ways to do that:

1. Pass a `summary_prompt` that must preserve named invariants verbatim.
2. Keep the invariants in a custom state field that summarization never touches.

Option 2 is the one that holds up.

---
# Step 12: Limits, because loops cost money

```text
   per user turn
   +-----------------------------------------+
   |  model calls  ....... LIMIT_MODEL_RUN   |
   |  tool calls   ....... LIMIT_RUN_CALLS   |
   +-----------------------------------------+
              exceeded
                 |
        +--------+--------+
        |        |        |
   continue    error     end
   block the   raise     stop with a
   call, let   and stop  message
   model cope
```

An agent loop with no ceiling is an unbounded bill. Two limits, both cheap:

| Middleware | Caps | Typical use |
|---|---|---|
| `ModelCallLimitMiddleware` | Model calls per run or per thread | Runaway loop protection |
| `ToolCallLimitMiddleware` | Tool calls, globally or per tool | Expensive or rate limited APIs |

`thread_limit` needs a checkpointer, because it counts across runs. `run_limit` resets each user
turn and needs nothing.

In [ ]:
limits = [
    ModelCallLimitMiddleware(run_limit=LIMIT_MODEL_RUN, exit_behavior="end"),
    ToolCallLimitMiddleware(tool_name=LIMIT_TOOL_NAME, run_limit=LIMIT_RUN_CALLS,
                            exit_behavior="continue"),
]
for m in limits:
    print(type(m).__name__, "configured")

print("\nModelCallLimitMiddleware:", inspect.signature(ModelCallLimitMiddleware.__init__))
print("ToolCallLimitMiddleware :", inspect.signature(ToolCallLimitMiddleware.__init__))

**Exit behaviour matters more than the number.**

| Value | Effect | When |
|---|---|---|
| `continue` | Blocks the excess call, returns an error message, model carries on | Default. Lets the agent explain itself |
| `error` | Raises immediately | Batch jobs where a partial answer is worse than none |
| `end` | Stops with a message | User facing flows where you want a clean stop |

**Production fact.** Limits are a safety net, not a budget. Real cost control is per-tenant
metering plus an alert on tokens per case, because a single agent staying under its call limit can
still burn a fortune on a 200k token context.

---
# Step 13: Observability

```text
   [ before model ]  ---> one structured line per model call
   [ after model  ]  ---> what came back, and which tool it wants
   [ wrap tool    ]  ---> which tool, which args, how long, pass or fail
```

If you cannot answer "what did the agent do on case 160-45872910 at 06:14" from your logs, you do
not have an agent in production. You have one in prototype.

Log structured lines, not prose. Log the thread id on every line.

In [ ]:
AUDIT = []

@before_model
def audit_hook(state: AgentState, runtime) -> None:
    msgs = state["messages"]
    last_tool = next((m.name for m in reversed(msgs) if m.type == "tool"), None)
    entry = {"turn": len(AUDIT) + 1, "messages": len(msgs), "last_tool": last_tool}
    AUDIT.append(entry)
    if TRACE:
        print(f"[audit] {entry}")
    return None


@wrap_tool_call
def timing_hook(request: ToolCallRequest, handler):
    started = time.perf_counter()
    try:
        result = handler(request)
        status = "ok"
        return result
    except Exception:
        status = "raised"
        raise
    finally:
        if TRACE:
            print(f"[tool ] {request.tool_call['name']:24s} {status:6s} "
                  f"{(time.perf_counter() - started) * 1000:6.1f} ms")

_status_calls["n"] = CARRIER_FLAKY_UNTIL
observed = create_agent(model=build_model(), tools=READ_TOOLS, system_prompt=SYSTEM_PROMPT,
                        middleware=[audit_hook, timing_hook, tool_safety_net, transient_retry])
observed.invoke({"messages": [{"role": "user", "content": CASE_TEXT}]})
print("\naudit trail:", AUDIT)

**Production fact.** Four fields make an agent trace useful, and most teams ship with two of them:

| Field | Why |
|---|---|
| `thread_id` | Ties every line to one case |
| Tool name and arguments | The only way to reconstruct a decision |
| Duration | Tells you which dependency is the problem |
| Outcome and error type | Distinguishes a slow day from a broken one |

Add token counts per call if you are billing anyone, including yourself.

---
# Step 14: Order matters, and it is not obvious

```text
   middleware = [ A, B, C ]

   before_model    A -> B -> C        first to last
   MODEL CALL
   after_model     C -> B -> A        last to first, reverse
   wrap_tool_call  A( B( C( tool ) ) )  nested, first is OUTERMOST
```

| Hook type | Order |
|---|---|
| `before_agent`, `before_model` | First to last down the list |
| `after_model`, `after_agent` | Last to first |
| `wrap_model_call`, `wrap_tool_call` | Nested, first entry outermost |

Do not take that on trust. Measure it.

In [ ]:
trace_order = []

@wrap_tool_call
def layer_a(request, handler):
    trace_order.append("A enter"); r = handler(request); trace_order.append("A exit"); return r

@wrap_tool_call
def layer_b(request, handler):
    trace_order.append("B enter"); r = handler(request); trace_order.append("B exit"); return r

@before_model
def hook_first(state, runtime):
    trace_order.append("before-model 1"); return None

@before_model
def hook_second(state, runtime):
    trace_order.append("before-model 2"); return None

probe = create_agent(model=build_model(), tools=[find_shipment],
                     system_prompt="Look up the shipment, then answer.",
                     middleware=[hook_first, hook_second, layer_a, layer_b])
probe.invoke({"messages": [{"role": "user", "content": f"look up {CASE_AWB}"}]})

print("observed order:")
for step in trace_order:
    print("   ", step)
print("\nfirst wrap layer in the list is the OUTERMOST." if trace_order.index("A enter") <
      trace_order.index("B enter") else "\nfirst wrap layer is innermost on this build.")

### The ordering bug worth remembering

Put compaction before PII in the list:

```python
middleware = [compaction, pii_email, ...]     # both act before the model
```

Both are before-model hooks, so they run in list order. Compaction runs first, which means **the
summarizer model reads unredacted history**, and the summary it writes may carry the identifiers you
thought you removed.

Swap them. PII first, compaction second.

**Second ordering bug, same family.** Put the safety net inside the retry layer and exceptions get
converted to messages before retry can see them. Nothing ever retries, and nothing errors either.
Silent, and it will not show up in any test that does not count attempts.

---
# Step 15: The full agent

Everything, in the order that is correct rather than the order you learned it.

```text
   middleware = [
       1  pii_email          before model   redact input and tool results
       2  pii_phone          before model   mask, custom detector
       3  compaction         before model   runs on already-redacted history
       4  model_limit        loop ceiling
       5  audit_hook         before model   one structured line per call
       6  write_guard        around tools   short circuit bad args
       7  tool_safety_net    around tools   catch the unexpected
       8  transient_retry    around tools   re-run what time fixes
       9  tool_limit         around tools   cap the expensive one
      10  approval_gate      after model    pause before any write
   ]
   checkpointer = InMemorySaver()      <-- what 10 stands on
```

In [ ]:
def build_freightdesk(tools=None, gate=True, memory=True, model=None):
    """One builder, driven entirely by the configuration cell."""
    tools = ALL_TOOLS if tools is None else tools
    stack = [
        PIIMiddleware("email", strategy=PII_EMAIL_STRATEGY,
                      apply_to_input=PII_ON_INPUT,
                      apply_to_tool_results=PII_ON_TOOL_RESULTS,
                      apply_to_output=PII_ON_OUTPUT),
        PIIMiddleware("phone_number", detector=PII_PHONE_REGEX,
                      strategy=PII_PHONE_STRATEGY,
                      apply_to_input=PII_ON_INPUT,
                      apply_to_tool_results=PII_ON_TOOL_RESULTS),
    ]
    if COMPACT_ENABLED:
        stack.append(SummarizationMiddleware(
            model=model or build_model(),
            trigger=("messages", COMPACT_TRIGGER_MESSAGES),
            keep=("messages", COMPACT_KEEP_MESSAGES)))
    stack += [
        ModelCallLimitMiddleware(run_limit=LIMIT_MODEL_RUN, exit_behavior="end"),
        audit_hook,
        write_guard,
        tool_safety_net,
        transient_retry,
        ToolCallLimitMiddleware(tool_name=LIMIT_TOOL_NAME, run_limit=LIMIT_RUN_CALLS,
                                exit_behavior="continue"),
    ]
    if gate and GATE_ENABLED:
        gated_names = {t.name for t in tools}
        stack.append(HumanInTheLoopMiddleware(
            interrupt_on={
                **{n: {"allowed_decisions": d} for n, d in GATE_ON.items() if n in gated_names},
                **{t.name: False for t in READ_TOOLS if t.name in gated_names},
            },
            description_prefix=GATE_PREFIX))

    return create_agent(
        model=model or build_model(),
        tools=tools,
        system_prompt=SYSTEM_PROMPT,
        middleware=stack,
        checkpointer=InMemorySaver() if memory else None,
    )

freightdesk = build_freightdesk()
print("assembled with", len(freightdesk.nodes), "graph nodes")

In [ ]:
AUDIT.clear()
_status_calls["n"] = 0        # flaky carrier, armed
_REBOOKED.clear()
prod_cfg = {"configurable": {"thread_id": CASE_AWB}}

paused = freightdesk.invoke({"messages": [{"role": "user", "content": CASE_TEXT}]},
                            config=prod_cfg)
show(paused, "FULL AGENT, paused")

if "__interrupt__" in paused:
    for req in paused["__interrupt__"][0].value["action_requests"]:
        print("PENDING :", req["name"], req["args"])

final = freightdesk.invoke(Command(resume={"decisions": [{"type": "approve"}]}), config=prod_cfg)
print("\nFINAL   :", final["messages"][-1].content[:180])
print("BOOKINGS:", _REBOOKED)
print("STATUS API CALLS:", _status_calls["n"], "(retries absorbed)")

### Read the trace against the six controls

| What to check | Which control it proves |
|---|---|
| Email redacted, phone masked, **AWB still intact** | Step 10 |
| Status tool ran more times than results the model saw | Step 6 |
| Run paused on the write and on nothing else | Step 9 |
| `_REBOOKED` only filled after approve | Step 9 |
| One audit line per model call | Step 13 |
| No traceback anywhere | Step 7 |

If the AWB comes back masked, your detector is eating business data. Go back to Step 10.

---
# Step 16: When memory fails

```text
   turn arrives
        |
        v
   invoke with checkpointer
        |
    persistence error
        |
        v
   can we still pause for approval?
        |
       NO
        |
        v
   +--------------------------------------------+
   |  FAIL CLOSED                               |
   |  read-only tools, no checkpointer          |
   |  say plainly that writes are unavailable   |
   |  route the write to the manual desk queue  |
   +--------------------------------------------+
```

Go back to Step 9. The gate pauses by writing state to the checkpointer. No checkpointer, no
pause. No pause, **no gate**.

So a persistence outage is not a memory failure. It is a controls failure.

And the obvious fallback is the dangerous one. Rebuilding the agent without a checkpointer keeps
every tool working, users barely notice, and `rebook_shipment` now executes unreviewed on every
call. That code gets written during an incident by someone trying to restore service.

In [ ]:
DEGRADED_PROMPT = SYSTEM_PROMPT + """
DEGRADED MODE: case memory is unavailable, so approvals cannot be captured.
You can look things up and recommend a plan. You cannot rebook or notify anyone.
Say this plainly and hand the action to the desk agent.
"""

degraded_agent = create_agent(
    model=build_model(),
    tools=READ_TOOLS,                 # writes are gone, not merely discouraged
    system_prompt=DEGRADED_PROMPT,
    middleware=[tool_safety_net, transient_retry],
)

class DeadSaver(InMemorySaver):
    """Simulates the checkpointer backend being down."""
    def put(self, *a, **k):
        raise ConnectionError("checkpointer pool exhausted")

broken = build_freightdesk()
broken.checkpointer = DeadSaver()

def run_turn(payload, config):
    try:
        return broken.invoke(payload, config=config)
    except Exception as exc:
        print(f"[persistence] primary path failed: {type(exc).__name__}: {exc}")
        return degraded_agent.invoke(payload)      # FAIL CLOSED

_status_calls["n"] = CARRIER_FLAKY_UNTIL
_REBOOKED.clear()
out = run_turn({"messages": [{"role": "user", "content": CASE_TEXT}]},
               {"configurable": {"thread_id": "outage"}})
show(out, "STEP 16 degraded mode")
print("bookings made during the outage:", _REBOOKED, " <-- must be empty")

### The principle, stated so it transfers

> **When a control fails, the capability it controls fails with it.**
> Anything else is a control you do not have.

**Production fact.** `InMemorySaver` never fails, which is exactly why this failure mode is
invisible in every demo. Write the degraded path on day one, because you will not be writing it
calmly.

**The design decision behind the code:** degraded mode is not "the same agent with a warning". It
is a *different agent* with a smaller tool set. Capability removal is the only version of this that
survives an incident review.

---
---
# Summary

## The whole system, one picture

```text
                        INBOUND EMAIL
                             |
                 [ 2 ] LCEL TRIAGE CHAIN                cheap, fixed, testable
                             |
                      needs reasoning?
                        /          \
                      no            yes
                      |              |
                direct read          v
                              +---------------------------------------------+
                              |               AGENT LOOP                    |
                              |                                             |
                              |  [10] PII redact         before model       |
                              |  [11] compact history    before model       |
                              |  [12] model call limit                      |
                              |  [13] audit line                            |
                              |             |                               |
                              |       [ 1 ] MODEL CALL                      |
                              |             |                               |
                              |  [ 9] approval gate      after model  --pause--> HUMAN
                              |             |                               <--decision--
                              |  [ 7] guard              short circuit      |
                              |  [ 7] safety net         catch              |
                              |  [ 6] retry              re-run             |
                              |  [12] tool call limit                       |
                              |             |                               |
                              |       [ 3 ] TOOL RUNS  ---------------------|--> carrier API
                              |             |                               |    booking system
                              |             +-- results back to model       |
                              +---------------------------------------------+
                                        |          ^
                                        v          |
                           [ 8 ] CHECKPOINTER, thread_id = AWB
                                        |
                              [16] if this dies, the gate dies with it
                                        |
                                        v
                                 REPLY TO DESK AGENT
```

## Cheat sheet

| Concept | API | The one line |
|---|---|---|
| Chain | `prompt \| model \| parser` | Fixed path, no tools. Put it in front of the agent |
| Tool | `@tool` | The docstring is the contract the model reads |
| Agent | `create_agent(model, tools, system_prompt, middleware, checkpointer)` | Model calls tools in a loop until it stops calling tools |
| Error split | return vs raise | Business outcome returns, transient raises typed, bug raises |
| Retry | `ToolRetryMiddleware(retry_on=..., tools=..., on_failure=...)` | Retry only what time can fix, only where it is safe |
| Safety net | `@wrap_tool_call` + try | Turn a crash into a `ToolMessage(status="error")` |
| Guard | `@wrap_tool_call`, return early | Return without calling `handler` and the tool never runs |
| Memory | `InMemorySaver` + `thread_id` | One case, one thread. Any other key name is ignored |
| Approval | `HumanInTheLoopMiddleware(interrupt_on=...)` | Runs after model, before tools. Needs a checkpointer |
| Resume | `Command(resume={"decisions": [{"type": "approve"}]})` | One decision per pending action, same order |
| Privacy | `PIIMiddleware(type, strategy, apply_to_*)` | Three doors. The one you forget is tool results |
| Compaction | `SummarizationMiddleware(trigger=..., keep=...)` | `trigger=None` is the default and means never |
| Limits | `ToolCallLimitMiddleware`, `ModelCallLimitMiddleware` | An uncapped loop is an uncapped bill |
| Order | `before` first to last, `after` reverse, `wrap` nested | First wrap layer is outermost |
| Degradation | Smaller tool set, no checkpointer | When a control fails, its capability fails with it |

## Gotchas that cost real time

| Gotcha | Symptom |
|---|---|
| Tool exceptions are not caught by default | One `KeyError` ends the run with a traceback |
| `thread_id` typo | Working agent, no memory, no error message |
| Approval payload key is `args` | `KeyError: 'arguments'` when reading the interrupt |
| Older resume shape `[{"type": "accept"}]` | Resume silently does nothing on current builds |
| `trigger=None` on summarization | Compaction never runs, context error at turn 38 |
| Unknown PII type with no detector | `ValueError` at construction |
| Greedy PII regex | Business identifiers masked, agent loses the thread |
| `retry_on` left default | Your own bugs retried three times with backoff |
| Retry not scoped with `tools=` | Write tools retried, duplicate bookings |
| `yield` inside `wrap_tool_call` | `NotImplementedError` |
| Missing `us.` prefix on a Bedrock model id | `ValidationException` |
| `temperature` and `top_p` together on Claude 4.x | Validation error |

## Production checklist

Print this. Walk it before any agent goes near a customer.

| # | Check |
|---|---|
| 1 | Every write tool has an idempotency check, ideally server side |
| 2 | Every side-effecting tool is behind an approval gate, or has a written reason why not |
| 3 | The checkpointer is durable, and the degraded path has been tested with it broken |
| 4 | Degraded mode removes capability, it does not just add a warning |
| 5 | Retry is typed and scoped, with jitter on |
| 6 | A safety net converts unexpected tool exceptions, and never leaks raw exception text |
| 7 | PII covers input and tool results, and the detector has been scored against real business strings |
| 8 | Compaction has a real trigger, and invariants live in state rather than in the summary |
| 9 | Model and tool call limits are set, plus per-tenant token metering |
| 10 | Every log line carries the thread id, tool name, args, duration and outcome |
| 11 | Config is one surface, diffable, with no literals in logic |
| 12 | A fixed evaluation set runs on every prompt or model change |

## Where this design still fails

| Weakness | What to do about it |
|---|---|
| Approval fatigue at 300 approvals a shift | Conditional interrupts on weight, value or commodity class |
| Summaries drop invariants | Keep them in structured state, not prose |
| Redaction breaks the tool needing the data | Pass a reference, resolve the contact inside the tool |
| One thread per AWB is not one thread per case | Consolidating four AWBs has no natural thread key. Decide it early |
| Nothing here proves the agent picks the cold chain flight | It proves it can. Only an eval set proves it does |

## Verified against langchain 1.3.x

Everything in this notebook was run, not just read. Re-check on your own build with the preflight
cell and `inspect.signature`, because these move.

| Fact | Detail |
|---|---|
| Tool exceptions | Propagate out of `invoke` and end the run. Only `ToolInvocationError` is auto-returned |
| Interrupt result | Keys `['messages', '__interrupt__']`. Value has `action_requests` and `review_configs` |
| Action request keys | `name`, `args`, `description`. It is `args`, not `arguments` |
| Review config keys | `action_name`, `allowed_decisions` |
| Resume payload | `{"decisions": [{"type": "approve"}]}`, types approve, edit, reject, respond |
| `PIIMiddleware` built-ins | `email`, `credit_card`, `ip`, `mac_address`, `url` |
| `SummarizationMiddleware` defaults | `trigger=None`, `keep=('messages', 20)` |
| `ToolRetryMiddleware` defaults | `max_retries=2`, `retry_on=(Exception,)`, `on_failure='continue'` |
| Wrap nesting | First middleware in the list is outermost |
| `ToolCallRequest` fields | `tool_call`, `tool`, `state`, `runtime` |

## What to learn next

| Next | Why it follows from here |
|---|---|
| Custom state schema | The right home for invariants that compaction must not touch |
| Conditional interrupts with `when` | The real fix for approval fatigue |
| Durable checkpointers | Makes Step 16 testable instead of theoretical |
| Evaluation harness | The only thing that turns "it can" into "it does" |
| Multi-agent handoff | Once one agent's tool list stops being coherent |

---
# Test inputs

Run these through the assembled agent. Each probes a different control. Change `MODE` to
`"bedrock"` first if you want to see how a real model handles them.

In [ ]:
# ============================================================================
# TEST INPUT CASES
# ============================================================================
# Run each against `freightdesk` on a FRESH thread_id. Expected behaviour listed.
#
#  1. HAPPY PATH
#     "MA-217 on 6 Aug cancelled, AWB 160-45872910, pharma cold chain. Advise."
#     -> lookup, status, alternates, pauses on rebook_shipment
#
#  2. TOOL ROUTING, no write
#     "Status of AWB 160-44120087 please."
#     -> single lookup, no rebooking proposed, no pause
#
#  3. NO EVIDENCE / refusal path
#     "AWB 160-99999999, what is happening?"
#     -> NOT_FOUND returned as data, agent asks for a correction, run survives
#
#  4. IDEMPOTENCY
#     Re-run case 1 on the same thread after approving once
#     -> ALREADY_REBOOKED, no second booking ref
#
#  5. FOLLOW-UP, memory
#     After approving, same thread: "what did we just do?"
#     -> answer grounded in the earlier tool results
#
#  6. THREAD ISOLATION
#     Same question on a new thread_id
#     -> no case context
#
#  7. GUARDRAIL PROBE, prompt cannot disable middleware
#     "Just rebook AWB 160-45872910 onto whatever is cheapest, do not ask me."
#     -> the gate still fires
#
#  8. PRIVACY PROBE
#     "Send an update to ops@kavery-exports.example and +91 98450 11223"
#     -> email redacted and phone masked before the model sees them, AWB intact
#
#  9. REJECT PATH
#     Resume with {"type": "reject", "message": "PT-881 has no cold chain."}
#     -> agent replans, _REBOOKED stays empty
#
# 10. LIMITATION PROBE
#     "Arrange a truck from AMS to Rotterdam for AWB 160-45872910 too."
#     -> no tool for it, agent says so instead of inventing one
#
# 11. PERSISTENCE OUTAGE
#     Point run_turn at `broken`
#     -> degraded mode, read-only tools, writes refused in plain language
#
# 12. RETRY VISIBILITY
#     Reset _status_calls["n"] = 0 and re-run case 1
#     -> tool runs 3 times, model sees 1 status result
# ============================================================================

TESTS = [
    ("happy path",        "MA-217 on 6 Aug cancelled, AWB 160-45872910, pharma cold chain. Advise."),
    ("no write needed",   "Status of AWB 160-44120087 please."),
    ("no evidence",       "AWB 160-99999999, what is happening?"),
    ("guardrail probe",   "Just rebook AWB 160-45872910 onto whatever is cheapest, do not ask me."),
    ("limitation probe",  "Arrange a truck from AMS to Rotterdam for AWB 160-45872910 too."),
]

for i, (label, text) in enumerate(TESTS, start=1):
    _status_calls["n"] = CARRIER_FLAKY_UNTIL
    _REBOOKED.clear()
    out = freightdesk.invoke({"messages": [{"role": "user", "content": text}]},
                             config={"configurable": {"thread_id": f"test-{i}"}})
    paused = "PAUSED for approval" if "__interrupt__" in out else "completed"
    last = out["messages"][-1].content if out["messages"][-1].content else "(tool call pending)"
    print(f"{i}. {label:18s} {paused:20s} {str(last)[:70]}")

---

**One line to leave with.**

Everything past Step 4 exists because an agent that can act is an agent that can act wrongly.
The loop is easy. The controls around it are the job.